# Phase 4 — Training

The big one. Before any real run we earn confidence in stages:

1. **Noam LR schedule** — plot it, understand warmup → inverse-sqrt.
2. **Label smoothing** — confirm the hand-written math matches the built-in.
3. **Overfit one batch** — the make-or-break test. Loss must crash toward ~0. If it
   can't, the model is broken and we fix it *before* anything else.
4. **Overfit ~10 batches** — same expectation.
5. **A small real run** — live loss / LR / grad-norm curves, attention before-vs-after,
   and a few greedy decodes.

Reusable primitives live in `transformer/train.py`; here we use them inline so every
step is visible. That same `fit()` is what the A100 script will call later.

In [ ]:
# Bootstrap: put the repo root on sys.path so `import transformer` works from notebooks/
import sys, pathlib
ROOT = pathlib.Path.cwd()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

In [ ]:
import math, time
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.optim import Adam

from transformer.config import ModelConfig, TrainConfig
from transformer.tokenizer import train_joint_bpe, PAD_ID, BOS_ID, EOS_ID
from transformer.data import load_multi30k, make_dataloader, collate_fn, TranslationDataset
from transformer.transformer import Transformer
from transformer.train import lr_at, LabelSmoothingLoss, cross_entropy_loss, grad_global_norm, fit

def get_device():
    if torch.cuda.is_available(): return torch.device("cuda")
    if torch.backends.mps.is_available(): return torch.device("mps")
    return torch.device("cpu")
device = get_device()
torch.manual_seed(0)
print("device:", device)

data, online = load_multi30k()
tokenizer = train_joint_bpe(data["train"], vocab_size=10000)
VOCAB = tokenizer.get_vocab_size()
print("online:", online, "| vocab:", VOCAB)

## 1. The Noam learning-rate schedule

`lr = d_model^-0.5 * min(step^-0.5, step * warmup^-1.5)`.

Two regimes that cross exactly at `step == warmup`: a **linear warmup** (the
`step * warmup^-1.5` term wins early) then **inverse-sqrt decay** (`step^-0.5` wins
after). Warmup stops huge early updates from a randomly-initialized model blowing up;
the slow decay then lets it settle. Stare at the curve until that story is obvious.

In [ ]:
steps = range(1, 100_000, 100)
for d_model, w in [(256, 4000), (512, 4000), (256, 8000)]:
    plt.plot(list(steps), [lr_at(s, d_model, w) for s in steps],
             label=f"d_model={d_model}, warmup={w}")
plt.axvline(4000, color="k", ls=":", lw=1)
plt.xlabel("step"); plt.ylabel("learning rate"); plt.legend()
plt.title("Noam schedule — peak at warmup, then inverse-sqrt decay")
plt.tight_layout(); plt.show()
print("peak lr (d=256, warmup=4000):", lr_at(4000, 256, 4000))

## 2. Label smoothing — write the math, then trust the built-in

Instead of a one-hot target we put `1-eps` on the true token and spread `eps` over the
vocab. This stops the model from becoming over-confident and improves BLEU (paper §5.4).
We wrote it by hand in `train.py`; here we confirm it equals `F.cross_entropy(...,
label_smoothing=eps)` so we can use the fast built-in with confidence.

**Important consequence:** smoothing puts a *floor* on the loss (you can never reach 0,
because the target itself isn't one-hot). That's why the overfit test below turns
smoothing OFF.

In [ ]:
torch.manual_seed(1)
logits = torch.randn(40, VOCAB); target = torch.randint(0, VOCAB, (40,)); target[:6] = PAD_ID
hand = LabelSmoothingLoss(pad_id=PAD_ID, smoothing=0.1)(logits, target)
builtin = F.cross_entropy(logits, target, ignore_index=PAD_ID, label_smoothing=0.1)
print(f"hand-written : {hand.item():.6f}")
print(f"torch builtin: {builtin.item():.6f}")
print("match:", torch.allclose(hand, builtin, atol=1e-5))

## 3. Overfit ONE batch — the make-or-break test

Take a single batch, turn off dropout and label smoothing, and train on it for a few
hundred steps. A correctly-wired model **must** drive the loss toward ~0 (it just has
to memorize one batch). **If it plateaus, stop and debug — do not start real training.**

In [ ]:
def fresh_model(dropout=0.0):
    cfg = ModelConfig.smoke(src_vocab_size=VOCAB, tgt_vocab_size=VOCAB)
    cfg.dropout = dropout
    return Transformer(cfg).to(device)

def overfit(model, batches, steps, lr=1e-3):
    model.train()
    opt = Adam(model.parameters(), lr=lr, betas=(0.9, 0.98), eps=1e-9)
    losses = []
    for s in range(steps):
        b = batches[s % len(batches)]
        opt.zero_grad(set_to_none=True)
        logits = model(b["src"], b["tgt_in"], b["src_pad"], b["tgt_pad"])
        loss = cross_entropy_loss(logits, b["tgt_out"], PAD_ID, smoothing=0.0)  # no floor
        loss.backward(); opt.step()
        losses.append(loss.item())
    return losses

loader = make_dataloader(data["train"], tokenizer, batch_size=16, shuffle=True, max_len=40)
one_batch = {k: v.to(device) for k, v in next(iter(loader)).items()}

m1 = fresh_model()
losses1 = overfit(m1, [one_batch], steps=300, lr=1e-3)
plt.plot(losses1); plt.xlabel("step"); plt.ylabel("loss")
plt.title("Overfit one batch — should crash toward 0"); plt.tight_layout(); plt.show()
print(f"start loss {losses1[0]:.3f}  ->  final loss {losses1[-1]:.4f}")
assert losses1[-1] < 0.15, "model failed to overfit one batch — it is BROKEN, debug first"
print("model can memorize one batch ✓ — wiring is correct")

## 4. Overfit ~10 batches

A slightly harder memorization. Loss should still fall steeply (it won't hit 0 as fast
as a single batch, but it should get small).

In [ ]:
it = iter(make_dataloader(data["train"], tokenizer, batch_size=16, shuffle=True, max_len=40))
ten = [{k: v.to(device) for k, v in next(it).items()} for _ in range(10)]

m10 = fresh_model()
losses10 = overfit(m10, ten, steps=600, lr=1e-3)
plt.plot(losses10); plt.xlabel("step"); plt.ylabel("loss")
plt.title("Overfit 10 batches"); plt.tight_layout(); plt.show()
print(f"start {losses10[0]:.3f}  ->  final {losses10[-1]:.4f}")

## 5. A small real training run

Now train properly on a subset with the full recipe (dropout 0.1, label smoothing 0.1,
Noam schedule, grad clipping) using `fit()` from `train.py`. We keep it short so it runs
in seconds on this Mac — on the A100 you'd point the *same* `fit()` at the full data with
the first-run config and `max_steps=100_000`.

First we snapshot **cross-attention before training** so we can compare after.

In [ ]:
# Fixed example to watch attention align (source EN -> target DE).
ex_en, ex_de = data["validation"][0]
ex_ds = TranslationDataset([(ex_en, ex_de)], tokenizer, max_len=40)
ex_batch = {k: v.to(device) for k, v in collate_fn([ex_ds[0]]).items()}
src_toks = [tokenizer.id_to_token(i) for i in ex_batch["src"][0].tolist()]
tgt_toks = [tokenizer.id_to_token(i) for i in ex_batch["tgt_in"][0].tolist()]
print("EN:", ex_en, "\nDE:", ex_de)

def cross_attn_map(model):
    """Run the example through the model and grab decoder layer's cross-attention (head 0)."""
    layer = model.decoder.layers[-1].cross_attn
    layer.cache_attn = True
    model.eval()
    with torch.no_grad():
        model(ex_batch["src"], ex_batch["tgt_in"], ex_batch["src_pad"], ex_batch["tgt_pad"])
    layer.cache_attn = False
    return layer.last_attn[0, 0].cpu()  # (Tq=tgt, Tk=src)

model = fresh_model(dropout=0.1)
attn_before = cross_attn_map(model)

In [ ]:
train_cfg = TrainConfig(batch_size=32, warmup_steps=400, label_smoothing=0.1, log_every=50)
train_loader = make_dataloader(data["train"][:4000], tokenizer,
                               batch_size=train_cfg.batch_size, shuffle=True, max_len=40)
hist = fit(model, train_loader, train_cfg, device, max_steps=400)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
ax[0].plot(hist.step, hist.loss); ax[0].set_title("training loss"); ax[0].set_xlabel("step")
ax[1].plot(hist.step, hist.lr);   ax[1].set_title("learning rate (Noam)"); ax[1].set_xlabel("step")
ax[2].plot(hist.step, hist.grad_norm); ax[2].set_title("grad norm"); ax[2].set_xlabel("step")
plt.tight_layout(); plt.show()
print(f"loss {hist.loss[0]:.3f} -> {hist.loss[-1]:.3f}   (log(V)={math.log(VOCAB):.2f}; "
      f"ppl {hist.ppl[0]:.0f} -> {hist.ppl[-1]:.0f})")
print("grad norm stays finite (no explosion, not stuck at 0) — both would signal a bug")

## Attention before vs after

Even after a tiny run, cross-attention should move from roughly **uniform** (every
target token attends to everything) toward **structure** (some source positions get
more weight). On the A100 with a full run you'll see crisp source↔target alignment.

In [ ]:
attn_after = cross_attn_map(model)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, mat, title in [(ax[0], attn_before, "before"), (ax[1], attn_after, "after")]:
    im = a.imshow(mat, cmap="viridis", aspect="auto")
    a.set_title(f"cross-attn {title}"); a.set_xlabel("source (EN)"); a.set_ylabel("target (DE)")
    a.set_xticks(range(len(src_toks))); a.set_xticklabels(src_toks, rotation=90, fontsize=6)
    a.set_yticks(range(len(tgt_toks))); a.set_yticklabels(tgt_toks, fontsize=6)
plt.tight_layout(); plt.show()
print("rows ~uniform before, more peaked after (subtle after only 400 steps)")

## A few greedy decodes + save a checkpoint

Greedy decoding (argmax each step) on a couple of training examples — after only 400
steps on a subset these will be rough, but you should see *German-looking* output with
the right structure. Then we save a self-contained checkpoint (the bridge to Phase 5/6).

In [ ]:
import pathlib
for en, de in data["train"][:3]:
    b = {k: v.to(device) for k, v in collate_fn([TranslationDataset([(en, de)], tokenizer, 40)[0]]).items()}
    out = model.generate(b["src"], b["src_pad"], BOS_ID, EOS_ID, PAD_ID, max_len=40)
    print("EN  :", en)
    print("ref :", de)
    print("pred:", tokenizer.decode(out[0].tolist()))
    print()

pathlib.Path("../checkpoints").mkdir(exist_ok=True)
ckpt = {"model_state_dict": model.state_dict(),
        "model_config": model.cfg.__dict__,
        "tokenizer": tokenizer.to_str()}
torch.save(ckpt, "../checkpoints/nb_phase4.pt")
print("saved ../checkpoints/nb_phase4.pt  (state + config + tokenizer = self-contained)")

## Takeaways

- Warmup + inverse-sqrt is what keeps a from-scratch Transformer stable early.
- The **overfit-one-batch** test is the single highest-value check — it isolates "is the
  model wired correctly?" from "is it learning the task?".
- Loss falls, grad norm stays finite, attention starts to structure. All green.

**Next:** this is the point to lift `fit()` into a **GPU training script** (`scripts/train.py`)
for the A100 — same code, first-run config, full data, checkpoints — then `05_generation.ipynb`
(greedy vs beam search) and `06_evaluation.ipynb` (BLEU).